In [ ]:
import esda
import libpysal
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.plotting import configure_plotting
from analysis.src.spatial_models import se_coefficients
from analysis.src.constants import ADJUSTMENT_SETS, OUTCOMES, PRIMARY_ADJUSTMENT, RANDOM_STATE

In [ ]:
configure_plotting()

In [ ]:
OUT = experiment_paths('spatial/autocorrelation_and_error_models')
DISEASES = OUTCOMES
SPATIAL_ERROR_METHOD = 'sparse'
MORAN_PERMUTATIONS = 999

data = load_data()

np.random.seed(RANDOM_STATE)

In [ ]:
global_rows = []
for variable in ['walkability_score', *OUTCOMES]:
    sample = data.dropna(subset = [variable]).copy()

    weights = libpysal.weights.Queen.from_dataframe(sample, use_index = True)
    weights.transform = 'r'

    result = esda.Moran(sample[variable], weights, permutations = MORAN_PERMUTATIONS)
    global_rows.append({'variable': variable, 'n_lsoas': len(sample), 'moran_i': result.I, 'p_sim': result.p_sim})

pd.DataFrame(global_rows).to_csv(OUT.global_moran_csv, index = False)

In [ ]:
# Compare residual spatial autocorrelation and spatial-error sensitivity
spatial_rows, residual_maps = [], []
for adjustment, covariates in ADJUSTMENT_SETS.items():
    for outcome in DISEASES:
        model_data = data.loc[complete_case_mask(data, outcome, covariates)].copy()

        weights = libpysal.weights.Queen.from_dataframe(model_data, use_index = True)
        weights.transform = 'r'

        formula = f'np.log({outcome}) ~ walkability_score + ' + ' + '.join(covariates)
        glm = smf.glm(formula = formula, data = model_data, family = sm.families.Gaussian()).fit()
        residual_moran = esda.Moran(glm.resid_pearson, weights)

        coefficients, error_lambda = se_coefficients(
            np.log(model_data[outcome].to_numpy()),
            model_data[['walkability_score'] + covariates].to_numpy(),
            weights,
            method = SPATIAL_ERROR_METHOD,
        )

        glm_coefficient = glm.params['walkability_score']
        spatial_error_coefficient = coefficients[1]

        spatial_rows.append(
            {
                'adjustment': adjustment,
                'outcome': outcome,
                'n_lsoas': int(glm.nobs),
                'glm_coefficient': glm_coefficient,
                'spatial_error_coefficient': spatial_error_coefficient,
                'spatial_error_lambda': error_lambda,
                'spatial_error_method': SPATIAL_ERROR_METHOD,
                'spatial_error_attenuation_percent': 100 * (1 - abs(spatial_error_coefficient) / abs(glm_coefficient)),
                'residual_moran_i': residual_moran.I,
                'residual_moran_p_value': residual_moran.p_sim
            }
        )

        if adjustment == PRIMARY_ADJUSTMENT:
            model_data['residual'] = glm.resid_pearson
            residual_maps.append((outcome, model_data))

spatial_sensitivity_results = pd.DataFrame(spatial_rows)
spatial_sensitivity_results.to_csv(OUT.spatial_error_models_csv, index = False)

display(spatial_sensitivity_results.round(4))

In [ ]:
# Plot residual spatial patterns
fig, axes = plt.subplots(2, 4, figsize = (16, 8))

for ax, (outcome, model_data) in zip(axes.flat, residual_maps):
    model_data.plot(column = 'residual', cmap = 'coolwarm', linewidth = 0.05, edgecolor = 'white', ax = ax)
    ax.set(title = outcome.removeprefix('qof_'))
    ax.set_axis_off()

fig.tight_layout()
fig.savefig(OUT.residual_maps_png, dpi = 300, bbox_inches = 'tight')

plt.show()